In [27]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/keystroke_data.csv")

print(df.shape)
df.head()

(143379, 9)


,PARTICIPANT_ID,TEST_SECTION_ID,SENTENCE,USER_INPUT,KEYSTROKE_ID,PRESS_TIME,RELEASE_TIME,LETTER,KEYCODE
0,100001,1090979,Was wondering if you and Natalie connected?,Was wondering if you and Natalie connected?,51891207.0,1.473275e+12,1473275372663,SHIFT,16
1,100001,1090979,Was wondering if you and Natalie connected?,Was wondering if you and Natalie connected?,51891214.0,1.473275e+12,1473275372703,W,87
2,100001,1090979,Was wondering if you and Natalie connected?,Was wondering if you and Natalie connected?,51891219.0,1.473275e+12,1473275372903,a,65
3,100001,1090979,Was wondering if you and Natalie connected?,Was wondering if you and Natalie connected?,51891226.0,1.473275e+12,1473275372975,s,83
4,100001,1090979,Was wondering if you and Natalie connected?,Was wondering if you and Natalie connected?,51891231.0,1.473275e+12,1473275373079,,32


In [28]:
df = df.sort_values(by=["PARTICIPANT_ID", "PRESS_TIME"])

In [29]:
df["hold_time"] = df["RELEASE_TIME"] - df["PRESS_TIME"]

df["next_press"] = df.groupby("PARTICIPANT_ID")["PRESS_TIME"].shift(-1)
df["flight_time"] = df["next_press"] - df["RELEASE_TIME"]

df = df.dropna()

In [30]:
df["next_key"] = df.groupby("PARTICIPANT_ID")["LETTER"].shift(-1)

df["digraph"] = df["LETTER"] + "_" + df["next_key"]

df["digraph_time"] = df["next_press"] - df["PRESS_TIME"]

In [31]:
# Clean invalid values
df = df[df["flight_time"] > 0]
df = df[df["hold_time"] > 0]
df = df[df["digraph_time"] > 0]

# Remove extreme outliers
df = df[df["hold_time"] < 2000]
df = df[df["flight_time"] < 2000]
df = df[df["digraph_time"] < 2000]

print("After cleaning:", df.shape)

After cleaning: (91058, 15)


In [32]:
# Keep top 50 users with most data
top_users = df["PARTICIPANT_ID"].value_counts().head(50).index
df = df[df["PARTICIPANT_ID"].isin(top_users)]

print("Users after filtering:", df["PARTICIPANT_ID"].nunique())

Users after filtering: 50


In [34]:
window_size = 20

samples = []
labels = []

for user in df["PARTICIPANT_ID"].unique():
    user_df = df[df["PARTICIPANT_ID"] == user]
    
    for i in range(0, len(user_df) - window_size, window_size):
        chunk = user_df.iloc[i:i+window_size]
        
        features = [
    chunk["hold_time"].mean(),
    chunk["hold_time"].std(),
    chunk["hold_time"].min(),
    chunk["hold_time"].max(),

    chunk["flight_time"].mean(),
    chunk["flight_time"].std(),
    chunk["flight_time"].min(),
    chunk["flight_time"].max(),

    chunk["digraph_time"].mean(),
    chunk["digraph_time"].std()
]
        
        samples.append(features)
        labels.append(user)

features_df = pd.DataFrame(samples, columns=[
    "hold_mean", "hold_std", "hold_min", "hold_max",
    "flight_mean", "flight_std", "flight_min", "flight_max",
    "digraph_mean", "digraph_std"
])

features_df["PARTICIPANT_ID"] = labels

In [35]:
features_df.to_csv("../data/features.csv", index=False)

print("✅ Features saved successfully")

✅ Features saved successfully
